In [1]:
from espn_api.football import League
import pandas as pd

In [2]:

# Replace with your actual info
league_id = 619156725
year = 2025  # or 2025 if it's set up for next season
espn_s2 = 'AEBSrahjPMfdLGpD6sDc6kYq3PVfwNcUX46fVlDTk9hgw4l%2BBaEvie%2BYxliYSwdMx9r0jWxAV7YA7KA5lftZr8NRjm9BZYkSOcwfz9e34lo0ZNPKjTrDLokpQSkmshPp96vgIBVeCazTCjkeYP%2BjBHCT3m85oxljPJ8JKOLJvn%2F0ykoQwxgkQx1atCIalowLDw%2FdLwBNAOTGYm1qJwdLSdqc7e4R12lZfY7gLi2CN7a5soLmA8B%2FewuvyTZsHyRHWw4c8XWk4zmq2XdYKUDAS9844%2FTmWflMzOhkNBXpmlwdBg%3D%3D'  # Leave blank if public
swid = '{C00A8DB8-3E66-4AE6-AF3A-1CEFE7D634AE}'             # Leave blank if public

# Create the league object
league = League(league_id=league_id, year=year, espn_s2=espn_s2, swid=swid)

# === Print league settings ===
print("League Name:", league.settings.name)
#print("Number of Teams:", league.settings.size)

# === Print team rosters ===
# for team in league.teams:
#     print(f"\nTeam: {team.team_name} | Abbreviation: {team.team_abbrev} | Owner: {team.owners[0]['firstName']}")
#     for player in team.roster:
#         print(f"  {player.name} - {player.position} - {player.stats} pts")



League Name: $200 PPR, Auction Draft


In [4]:
league = League(league_id=league_id, year=2025, espn_s2=espn_s2, swid=swid)

for team in league.teams:
    for player in team.roster:
        print(dir(player))
        break
    break

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'acquisitionType', 'active_status', 'avg_points', 'eligibleSlots', 'injured', 'injuryStatus', 'lineupSlot', 'name', 'onTeamId', 'percent_owned', 'percent_started', 'playerId', 'posRank', 'position', 'proTeam', 'projected_avg_points', 'projected_total_points', 'schedule', 'stats', 'total_points']


In [ ]:
import pandas as pd
from espn_api.football import League

# Initialize your league
league = League(league_id=league_id, year=2025, espn_s2=espn_s2, swid=swid)

players_data = []

# Loop through each team and player
for team in league.teams:
    for player in team.roster:
        players_data.append({
            'player_name': player.name,
        'points_scored': player.total_points,
        'points_avg': player.avg_points,
        'projected_points_total': player.projected_total_points,
        'projected_points_avg': player.projected_avg_points,
        'position': player.position
            
            
            #'draft_budget_value': player.draftAuctionValue if player.acquisitionType == "DRAFT" else None,
        })
free_agents = league.free_agents(size=350)
for player in free_agents:
    players_data.append({
        'player': player.name,
        'projPtsTotal': player.projected_total_points,
        'projPtsAvg': player.projected_avg_points,
        'position': player.position
})
# Create the DataFrame



# Optional: Filter only drafted players
# df = df[df['acquisition_type'] == 'DRAFT']

# print(df.head(20))


df = pd.DataFrame(players_data)
df['posRank'] = df.groupby('position')['projPtsAvg'].rank(ascending=False, method='min').astype(int)
#War Calc
# Example DataFrame


# Target ranks for reference points
target_ranks = {'RB': 38, 'WR': 50, 'QB': 17, 'TE': 15}

# Dynamically calculate reference points
reference_points = {}
for position, rank in target_ranks.items():
    ref_value = df.loc[(df['position'] == position) & (df['posRank'] == rank), 'projPtsTotal']
    reference_points[f"{position}{rank}"] = ref_value.iloc[0] if not ref_value.empty else 0

#print("Reference Points:", reference_points)

# Apply the formula
def calculate_value(row):
    position = row['position']
    total_points = row['projPtsTotal']
    
    # Retrieve the reference dynamically
    reference_key = f"{position}{target_ranks.get(position, 0)}"
    reference = reference_points.get(reference_key, 0)
    
    # Compute the value as per the formula
    return round((total_points - reference) / 44.2216666667, 3)

# df['WAR'] = df.apply(calculate_value, axis=1)
#df.head(20)
#df["WAR"] = df["WAR"]+10
df2 = pd.read_csv("dataWithCR_updated.csv")
#print(df)
update_dict = df.set_index('player')[['projPtsTotal', 'projPtsAvg', 'posRank']].to_dict('index')

# Update df2 row-by-row based on player match
for idx, row in df2.iterrows():
    player = row['player']
    if player in update_dict:
        for col in ['projPtsTotal', 'projPtsAvg', 'posRank']:
            df2.at[idx, col] = update_dict[player][col]
df2.to_csv("dataWithCR_updated.csv", index=False)

In [ ]:
print(dir(free_agents[0]))

In [ ]:
# free_agents = league.free_agents(size=200)
# for player in free_agents:
#     players_data.append({
#         'player_name': player.name,
#         'projected_points_total': player.projected_total_points,
#         'projected_points_avg': player.projected_avg_points,
#         'position': player.position
# })
# # Create the DataFrame

# df = pd.DataFrame(players_data)
    

# # Optional: Filter only drafted players
# # df = df[df['acquisition_type'] == 'DRAFT']

# print(df.head())
# df.to_csv("espnAPIData.csv")

In [ ]:
df2 = pd.read_csv('espnAPIData.csv')
df = pd.read_csv("TotalUpdated6.7.csv")
merged_df = df2.merge(
    df[['player', 'value', 'predicted_amount_paid']],
    on='player',
    how='left'  # keeps all players from df2
)

# Fill missing values with 0
merged_df[['value', 'predicted_amount_paid']] = merged_df[['value', 'predicted_amount_paid']].fillna(0)


In [ ]:
merged_df.head(20)
merged_df.to_csv("espnAPIData.csv")

In [ ]:
df = pd.read_csv("espnAPIData.csv")
df["averageRank"] = (df["espnPaid"]+df["predicted_amount_paid"])/2
df.to_csv("espnAPIData.csv")

In [ ]:
espn_data = pd.read_csv("espnAPIData.csv")
cr_data = pd.read_csv("CR.csv")

# Merge datasets on 'player'
merged_data = pd.merge(espn_data, cr_data[['player', 'start%', 'CR']], on='player', how='left')

# Ensure 'CR' and 'start%' are numeric, coercing errors to NaN
merged_data['CR'] = pd.to_numeric(merged_data['CR'], errors='coerce')
merged_data['start%'] = pd.to_numeric(merged_data['start%'], errors='coerce')

# Calculate median CR and start% by position
median_cr_by_position = merged_data.groupby('position')['CR'].median()
median_start_by_position = merged_data.groupby('position')['start%'].median()

# Fill missing CR values using the position-specific median CR
merged_data['CR'] = merged_data.apply(
    lambda row: 1.2 * median_cr_by_position[row['position']] if pd.isna(row['CR']) else row['CR'],
    axis=1
)

# Fill missing start% values using the position-specific median start%
merged_data['start%'] = merged_data.apply(
    lambda row: 0.8 * median_start_by_position[row['position']] if pd.isna(row['start%']) else row['start%'],
    axis=1
)

# Save the updated dataset to a new CSV file
merged_data.to_csv("dataWithCR.csv", index=False)

# Display a preview of the updated dataset
print(merged_data.head())

In [ ]:
df = pd.read_csv("dataWithCR.csv")

df["CR"] = round(df["CR"],3)
df["STDEV"] = df['CR'] *df['projPtsAvg']
df["STDEV"] = round(df["STDEV"],3)

df.to_csv("dataWithCR.csv")

In [ ]:
import pandas as pd
from espn_api.football import League

# === Initialize your league ===
league = League(league_id=league_id, year=2024, espn_s2=espn_s2, swid=swid)

players_data = []

# === Loop through all WRs from all teams ===
for team in league.teams:
    for player in team.roster:
        
        if player.position != "WR":
            continue

        actual_targets = 0
        projected_targets = 0
        total_actual_points = 0
        total_projected_points = 0
        
        # Loop through weekly stats (1–17 or 18, depending on league)
        for week_data in player.stats.values():
            actual = week_data.get("breakdown", {})
            projected = week_data.get("projected_breakdown", {})
            actual_targets += actual.get("receivingTargets", 0)
            projected_targets += projected.get("receivingTargets", 0)
            total_actual_points += week_data.get("points", 0)
            total_projected_points += week_data.get("projected_points", 0)
            
            

        players_data.append({
            "player_name": player.name,
            "position": player.position,
            "actual_targets": round(actual_targets),
            "projected_targets": round(projected_targets),
            "total_actual_points": round(total_actual_points, 2),
            "total_projected_points": round(total_projected_points, 2)
        })

# === Convert to DataFrame and sort ===
df = pd.DataFrame(players_data)
df = df.sort_values(by="projected_targets", ascending=False)

# === Save to CSV ===
df.to_csv("wr_targets_summary_2025.csv", index=False)

print("✅ CSV saved as wr_targets_summary_2025.csv")


In [ ]:
import pandas as pd
from espn_api.football import League

# Initialize your league
league = League(league_id=league_id, year=2025, espn_s2=espn_s2, swid=swid)

players_data = []

# Loop through each team and player
for team in league.teams:
    for player in team.roster:
        players_data.append({
            'player_name': player.name,
            'team': team.team_name
        })

# Create DataFrame with just player_name and team
df = pd.DataFrame(players_data, columns=['player_name', 'team'])

# Save to CSV
df.to_csv("players_on_teams.csv", index=False)

print("✅ players_on_teams.csv created successfully")


In [10]:
league = League(league_id=league_id, year=2025, espn_s2=espn_s2, swid=swid)

players_data = []

# Loop through each team and player
x=0
for team in league.teams:
    for player in team.roster:
        if (x==0): 
            print(player.stats[1]["points"])
            # print(player.stats[1]["projected_points"])
            # print(player.stats[2]["projected_points"])
            x=1
            print(player.name)
        players_data.append({
            'player_name': player.name,
            'team': team.team_name
        })

# Create DataFrame with just player_name and team
df = pd.DataFrame(players_data, columns=['player_name', 'team'])

# Save to CSV
df.to_csv("players_on_teams.csv", index=False)




KeyError: 1